# 🐍 PySpark Masterclass — Zoteo Department of Health
### Unity Catalog → Bronze → Silver → Gold Pipeline

**Emergi Mentors — Data Analytics Bootcamp**

---

**What you'll learn in this notebook:**

| # | Topic | PySpark Concept |
|---|-------|----------------|
| 1 | Reading from Unity Catalog | `spark.table()`, `spark.read` |
| 2 | Exploring DataFrames | `.show()`, `.display()`, `.printSchema()`, `.describe()` |
| 3 | Selecting Columns | `.select()`, `col()`, column aliases |
| 4 | Filtering Rows | `.filter()`, `.where()`, compound conditions |
| 5 | Adding & Transforming Columns | `.withColumn()`, `when/otherwise`, `cast()` |
| 6 | Handling Nulls | `.isNull()`, `.fillna()`, `.dropna()` |
| 7 | GroupBy & Aggregations | `.groupBy()`, `agg()`, `count()`, `avg()`, `sum()` |
| 8 | Joins | `.join()` — inner, left, anti |
| 9 | Window Functions | `Window.partitionBy()`, `rank()`, `row_number()` |
| 10 | Writing to Gold Layer | `.write.saveAsTable()`, creating views |
| 11 | Bonus: Visualisation-Ready Outputs | `.toPandas()`, display charts |

> **Prerequisites:** Unity Catalog lab completed (zoteo_health catalog with bronze tables loaded)

## ⚙️ Setup — Imports & Catalog Context

In [0]:
# Standard PySpark imports you'll use in every project
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, DateType, StringType

# Set our Unity Catalog context
spark.sql("USE CATALOG zoteo_health")
spark.sql("USE SCHEMA bronze")

print("✅ Connected to zoteo_health.bronze")

✅ Connected to zoteo_health.bronze


---
## 1️⃣ Reading Data from Unity Catalog

In Unity Catalog, every table has a **three-level path**: `catalog.schema.table`

PySpark gives you two ways to read a table:

In [0]:
# METHOD 1: spark.table() — reads a Unity Catalog table directly
# This is the most common way when working with managed/external tables
df_admissions = spark.table("zoteo_health.bronze.raw_hospital_admissions")

# METHOD 2: spark.read — reads from a file path (e.g., from a Volume)
# Useful when loading raw files that haven't been converted to tables yet
df_from_csv = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/zoteo_health/landing/raw_files/dim_hospitals.csv")
)

print(f"Admissions from table: {df_admissions.count():,} rows")
print(f"Hospitals from CSV:    {df_from_csv.count()} rows")

Admissions from table: 51,500 rows
Hospitals from CSV:    45 rows


In [0]:
# Let's load all our reference tables too — we'll need them later
df_hospitals = spark.table("zoteo_health.bronze.dim_hospitals")
df_lhd       = spark.table("zoteo_health.bronze.dim_local_health_districts")
df_ed        = spark.table("zoteo_health.bronze.raw_ed_presentations")
df_funding   = spark.table("zoteo_health.bronze.fact_abf_funding_allocation")
df_quality   = spark.table("zoteo_health.bronze.ref_data_quality_log")

print("✅ All 6 Bronze tables loaded into DataFrames")

✅ All 6 Bronze tables loaded into DataFrames


---
## 2️⃣ Exploring DataFrames

Before transforming data, you need to **understand its shape, schema, and quality**.
These are the 5 exploration commands you'll use daily:

In [0]:
# 2a. printSchema() — shows column names, data types, and nullability
# Think of it as the "blueprint" of your DataFrame
df_admissions.printSchema()

root
 |-- admission_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- lhd_code: string (nullable = true)
 |-- admission_date: string (nullable = true)
 |-- separation_date: date (nullable = true)
 |-- length_of_stay_days: integer (nullable = true)
 |-- admission_type: string (nullable = true)
 |-- drg_code: string (nullable = true)
 |-- patient_age: double (nullable = true)
 |-- patient_gender: string (nullable = true)
 |-- separation_mode: string (nullable = true)
 |-- abf_price_weight: double (nullable = true)
 |-- source_system: string (nullable = true)
 |-- extract_timestamp: timestamp (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
# 2b. show() — displays the first N rows as plain text
# Good for quick checks in logs/terminal
df_admissions.show(5, truncate=False)

+------------+----------+-----------+--------+--------------+---------------+-------------------+--------------+--------+-----------+--------------+----------------------+----------------+-------------+-------------------+-------------+
|admission_id|patient_id|hospital_id|lhd_code|admission_date|separation_date|length_of_stay_days|admission_type|drg_code|patient_age|patient_gender|separation_mode       |abf_price_weight|source_system|extract_timestamp  |_rescued_data|
+------------+----------+-----------+--------+--------------+---------------+-------------------+--------------+--------+-----------+--------------+----------------------+----------------+-------------+-------------------+-------------+
|ADM00000001 |P712554   |H0004      |LHD002  |2025-01-07    |2025-01-12     |5                  |Elective      |B70A    |48.0       |NULL          |Discharged Home       |1.5736          |iPM          |2025-01-13 00:00:00|NULL         |
|ADM00000002 |P225710   |H0025      |LHD008  |2025-0

In [0]:
# 2c. display() — Databricks-specific rich table (sortable, filterable)
# This is what you'll use most in notebooks
display(df_admissions.limit(10))

admission_id,patient_id,hospital_id,lhd_code,admission_date,separation_date,length_of_stay_days,admission_type,drg_code,patient_age,patient_gender,separation_mode,abf_price_weight,source_system,extract_timestamp,_rescued_data
ADM00000001,P712554,H0004,LHD002,2025-01-07,2025-01-12,5,Elective,B70A,48.0,null,Discharged Home,1.5736,iPM,2025-01-13T00:00:00.000Z,null
ADM00000002,P225710,H0025,LHD008,2025-02-21,2025-03-08,15,Newborn,D06A,46.0,M,Discharged Home,0.8488,iPM,2025-03-08T11:00:00.000Z,null
ADM00000003,P712024,H0027,LHD009,2025-03-26,2025-04-11,16,Elective,null,68.0,F,Discharged Home,0.7199,HOMER,2025-04-12T07:00:00.000Z,null
ADM00000004,P237235,H0017,LHD006,2025-02-20,2025-02-25,5,Elective,I03B,41.0,F,null,0.7218,iPM,2025-02-25T02:00:00.000Z,null
ADM00000005,P663750,H0011,LHD004,2025-01-10,2025-01-14,4,Emergency,I03A,9.0,M,Statistical Separation,0.299,iPM,2025-01-15T08:00:00.000Z,null
ADM00000006,P265409,H0024,LHD008,2025-02-06,2025-02-07,1,Elective,B70B,29.0,X,Discharged Home,1.246,iPM,2025-02-09T23:00:00.000Z,null
ADM00000007,P377312,H0011,LHD004,2025-01-18,2025-01-19,1,Emergency,E65B,20.0,X,Discharged Home,2.7898,HOMER,2025-01-20T13:00:00.000Z,null
ADM00000008,P313487,H0039,LHD013,2025-02-13,2025-02-15,2,Elective,I03A,53.0,X,Discharged Home,0.3689,iPM,2025-02-15T12:00:00.000Z,null
ADM00000009,P969634,H0041,LHD014,2025-02-24,2025-02-25,1,Emergency,E62A,54.0,F,Discharged Home,0.4468,HOMER,2025-02-25T21:00:00.000Z,null
ADM00000010,P839945,H0029,LHD010,2025-03-12,2025-03-16,4,Elective,E62A,38.0,M,Discharged Home,0.8153,Cerner,2025-03-18T22:00:00.000Z,null


In [0]:
# 2d. describe() — summary statistics for numeric columns
# Shows count, mean, stddev, min, max
display(df_admissions.describe())

summary,admission_id,patient_id,hospital_id,lhd_code,admission_date,length_of_stay_days,admission_type,drg_code,patient_age,patient_gender,separation_mode,abf_price_weight,source_system,_rescued_data
count,51500,51500,51500,51500,51500,51500,51500,48962,48978,48859,48850,51500,51500,0
mean,null,null,null,null,null,4.960116504854369,null,null,51.46616848380906,null,null,1.2821459009708662,null,null
stddev,null,null,null,null,null,7.204467308350469,null,null,21.660783618194397,null,null,1.0111781885543387,null,null
min,ADM00000001,P100012,H0001,LHD001,01/01/2025,-1,Elective,B70A,0.0,F,Died in Hospital,0.0545,Cerner,null
max,ADM00050000,P999990,H9999,LHD015,31/03/2025,292,Transfer,O60B,105.0,X,Transferred,24.3731,iPM,null


In [0]:
# 2e. Useful properties
print(f"Total rows:    {df_admissions.count():,}")
print(f"Total columns: {len(df_admissions.columns)}")
print(f"Column names:  {df_admissions.columns}")

# dtypes gives you a list of (column_name, data_type) tuples
print(f"\nData types:")
for col_name, dtype in df_admissions.dtypes:
    print(f"  {col_name:30s} → {dtype}")

Total rows:    51,500
Total columns: 16
Column names:  ['admission_id', 'patient_id', 'hospital_id', 'lhd_code', 'admission_date', 'separation_date', 'length_of_stay_days', 'admission_type', 'drg_code', 'patient_age', 'patient_gender', 'separation_mode', 'abf_price_weight', 'source_system', 'extract_timestamp', '_rescued_data']

Data types:
  admission_id                   → string
  patient_id                     → string
  hospital_id                    → string
  lhd_code                       → string
  admission_date                 → string
  separation_date                → date
  length_of_stay_days            → int
  admission_type                 → string
  drg_code                       → string
  patient_age                    → double
  patient_gender                 → string
  separation_mode                → string
  abf_price_weight               → double
  source_system                  → string
  extract_timestamp              → timestamp
  _rescued_data              

---
## 3️⃣ Selecting Columns

`.select()` is the PySpark equivalent of SQL's `SELECT` clause.
There are **three ways** to reference columns:

In [0]:
# WAY 1: String names (simplest, most common)
df_admissions.select("admission_id", "patient_id", "hospital_id").show(5)

+------------+----------+-----------+
|admission_id|patient_id|hospital_id|
+------------+----------+-----------+
| ADM00000001|   P712554|      H0004|
| ADM00000002|   P225710|      H0025|
| ADM00000003|   P712024|      H0027|
| ADM00000004|   P237235|      H0017|
| ADM00000005|   P663750|      H0011|
+------------+----------+-----------+
only showing top 5 rows


In [0]:
# WAY 2: col() function (required for transformations)
df_admissions.select(
    F.col("admission_id"),
    F.col("patient_age"),
    F.col("admission_type")
).show(5)

+------------+-----------+--------------+
|admission_id|patient_age|admission_type|
+------------+-----------+--------------+
| ADM00000001|       48.0|      Elective|
| ADM00000002|       46.0|       Newborn|
| ADM00000003|       68.0|      Elective|
| ADM00000004|       41.0|      Elective|
| ADM00000005|        9.0|     Emergency|
+------------+-----------+--------------+
only showing top 5 rows


In [0]:
# WAY 3: DataFrame["column"] syntax
df_admissions.select(
    df_admissions["admission_id"],
    df_admissions["patient_age"]
).show(5)

+------------+-----------+
|admission_id|patient_age|
+------------+-----------+
| ADM00000001|       48.0|
| ADM00000002|       46.0|
| ADM00000003|       68.0|
| ADM00000004|       41.0|
| ADM00000005|        9.0|
+------------+-----------+
only showing top 5 rows


In [0]:
# ALIASING — rename columns in the output (like SQL's AS)
display(
    df_admissions.select(
        F.col("hospital_id").alias("hospital"),
        F.col("patient_age").alias("age"),
        F.col("length_of_stay_days").alias("los_days"),
        F.col("abf_price_weight").alias("funding_weight")
    ).limit(10)
)

hospital,age,los_days,funding_weight
H0004,48.0,5,1.5736
H0025,46.0,15,0.8488
H0027,68.0,16,0.7199
H0017,41.0,5,0.7218
H0011,9.0,4,0.299
H0024,29.0,1,1.246
H0011,20.0,1,2.7898
H0039,53.0,2,0.3689
H0041,54.0,1,0.4468
H0029,38.0,4,0.8153


---
## 4️⃣ Filtering Rows

`.filter()` and `.where()` are **identical** — use whichever reads better.
This is your PySpark equivalent of SQL's `WHERE` clause.

In [0]:
# FILTER 1: Simple condition — Emergency admissions only
df_emergency = df_admissions.filter(F.col("admission_type") == "Emergency")
print(f"Emergency admissions: {df_emergency.count():,}")

Emergency admissions: 18,009


In [0]:
# FILTER 2: Compound conditions — use & (AND), | (OR), ~ (NOT)
# IMPORTANT: Each condition must be wrapped in parentheses!
df_filtered = df_admissions.filter(
    (F.col("admission_type") == "Emergency") &
    (F.col("patient_age") >= 65) &
    (F.col("length_of_stay_days") > 0)
)
print(f"Emergency patients aged 65+ with valid LOS: {df_filtered.count():,}")

Emergency patients aged 65+ with valid LOS: 4,223


In [0]:
# FILTER 3: isin() — match against a list of values
df_surgical = df_admissions.filter(
    F.col("admission_type").isin("Elective", "Emergency")
)
print(f"Elective + Emergency: {df_surgical.count():,}")

Elective + Emergency: 36,018


In [0]:
# FILTER 4: String matching with like, contains, startswith
df_dup = df_admissions.filter(F.col("admission_id").like("%_DUP"))
print(f"Duplicate records (suffix _DUP): {df_dup.count():,}")

Duplicate records (suffix _DUP): 1,500


In [0]:
# FILTER 5: between() for ranges
df_working_age = df_admissions.filter(
    F.col("patient_age").between(18, 64)
)
print(f"Working-age patients (18-64): {df_working_age.count():,}")

Working-age patients (18-64): 32,509


---
## 5️⃣ Adding & Transforming Columns

`.withColumn()` adds a new column or replaces an existing one.
Combined with `F.when().otherwise()` (PySpark's CASE WHEN), this is incredibly powerful.

In [0]:
# 5a. withColumn + literal value
df_with_source = df_admissions.withColumn("data_layer", F.lit("bronze"))
display(df_with_source.select("admission_id", "data_layer").limit(5))

admission_id,data_layer
ADM00000001,bronze
ADM00000002,bronze
ADM00000003,bronze
ADM00000004,bronze
ADM00000005,bronze


In [0]:
# 5b. withColumn + calculation
df_with_calc = df_admissions.withColumn(
    "los_category",
    F.when(F.col("length_of_stay_days") <= 0, "Invalid")
     .when(F.col("length_of_stay_days") <= 1, "Day Case")
     .when(F.col("length_of_stay_days") <= 3, "Short Stay")
     .when(F.col("length_of_stay_days") <= 7, "Medium Stay")
     .when(F.col("length_of_stay_days") <= 14, "Long Stay")
     .otherwise("Extended Stay")
)

display(
    df_with_calc
    .groupBy("los_category")
    .count()
    .orderBy("count", ascending=False)
)

los_category,count
Short Stay,13695
Medium Stay,12166
Day Case,9838
Long Stay,6363
Invalid,6040
Extended Stay,3398


In [0]:
# 5c. Type casting — convert string to date, int to double, etc.
df_casted = df_admissions.withColumn(
    "patient_age_double", F.col("patient_age").cast(DoubleType())
)

# 5d. String transformations
df_upper = df_admissions.withColumn(
    "admission_type_upper", F.upper(F.col("admission_type"))
).withColumn(
    "admission_type_lower", F.lower(F.col("admission_type"))
)
display(df_upper.select("admission_type", "admission_type_upper", "admission_type_lower").limit(5))

admission_type,admission_type_upper,admission_type_lower
Elective,ELECTIVE,elective
Newborn,NEWBORN,newborn
Elective,ELECTIVE,elective
Elective,ELECTIVE,elective
Emergency,EMERGENCY,emergency


In [0]:
# 5e. Date functions — extract year, month, day, etc.
df_dates = df_admissions.withColumn(
    "admission_year",  F.year(F.col("separation_date"))
).withColumn(
    "admission_month", F.month(F.col("separation_date"))
).withColumn(
    "day_of_week",     F.dayofweek(F.col("separation_date"))
)

display(df_dates.select("separation_date", "admission_year", "admission_month", "day_of_week").limit(10))

separation_date,admission_year,admission_month,day_of_week
2025-01-12,2025,1,1
2025-03-08,2025,3,7
2025-04-11,2025,4,6
2025-02-25,2025,2,3
2025-01-14,2025,1,3
2025-02-07,2025,2,6
2025-01-19,2025,1,1
2025-02-15,2025,2,7
2025-02-25,2025,2,3
2025-03-16,2025,3,1


In [0]:
# 5f. withColumnRenamed — rename without transforming
df_renamed = df_admissions.withColumnRenamed("abf_price_weight", "funding_weight")
print(f"'abf_price_weight' renamed to: {'funding_weight' in df_renamed.columns}")

'abf_price_weight' renamed to: True


---
## 6️⃣ Handling Null Values

Real-world data is messy. The Bronze layer has **~5% nulls** in several columns.
PySpark gives you three strategies:

In [0]:
# 6a. Check for nulls — count nulls per column
null_counts = df_admissions.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_admissions.columns]
)
display(null_counts)

admission_id,patient_id,hospital_id,lhd_code,admission_date,separation_date,length_of_stay_days,admission_type,drg_code,patient_age,patient_gender,separation_mode,abf_price_weight,source_system,extract_timestamp,_rescued_data
0,0,0,0,0,0,0,0,2538,2522,2641,2650,0,0,0,51500


In [0]:
# 6b. dropna() — remove rows with ANY null, or nulls in specific columns
df_no_nulls = df_admissions.dropna(subset=["drg_code", "patient_age", "patient_gender"])
print(f"Before dropna: {df_admissions.count():,}")
print(f"After dropna:  {df_no_nulls.count():,}")
print(f"Rows removed:  {df_admissions.count() - df_no_nulls.count():,}")

Before dropna: 51,500
After dropna:  44,196
Rows removed:  7,304


In [0]:
# 6c. fillna() — replace nulls with a default value
df_filled = df_admissions.fillna({
    "drg_code": "UNKNOWN",
    "patient_age": -1,
    "patient_gender": "U",
    "separation_mode": "Unknown"
})
# Verify: no more nulls in those columns
display(
    df_filled.select(
        [F.count(F.when(F.col(c).isNull(), c)).alias(c)
         for c in ["drg_code", "patient_age", "patient_gender", "separation_mode"]]
    )
)

drg_code,patient_age,patient_gender,separation_mode
0,0,0,0


---
## 7️⃣ GroupBy & Aggregations

This is where data becomes **insight**. `.groupBy().agg()` is the PySpark
equivalent of SQL's `GROUP BY` with aggregate functions.

In [0]:
# 7a. Simple groupBy + count
display(
    df_admissions
    .groupBy("admission_type")
    .count()
    .orderBy("count", ascending=False)
)

admission_type,count
Elective,18009
Emergency,18009
Maternity,6237
Transfer,5091
Newborn,4154


In [0]:
# 7b. Multiple aggregations with agg()
display(
    df_admissions
    .groupBy("admission_type")
    .agg(
        F.count("*").alias("total_admissions"),
        F.round(F.avg("length_of_stay_days"), 1).alias("avg_los"),
        F.round(F.avg("patient_age"), 1).alias("avg_age"),
        F.round(F.sum("abf_price_weight"), 2).alias("total_funding_weight"),
        F.min("length_of_stay_days").alias("min_los"),
        F.max("length_of_stay_days").alias("max_los")
    )
    .orderBy("total_admissions", ascending=False)
)

admission_type,total_admissions,avg_los,avg_age,total_funding_weight,min_los,max_los
Elective,18009,5.0,51.6,23114.77,-1,121
Emergency,18009,5.0,51.6,23089.9,-1,292
Maternity,6237,4.9,51.0,7906.53,-1,129
Transfer,5091,4.8,51.0,6673.52,-1,88
Newborn,4154,4.9,51.7,5245.8,-1,221


In [0]:
# 7c. groupBy on multiple columns
display(
    df_admissions
    .filter(F.col("patient_gender").isNotNull())
    .groupBy("admission_type", "patient_gender")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("length_of_stay_days"), 1).alias("avg_los")
    )
    .orderBy("admission_type", "patient_gender")
)

admission_type,patient_gender,count,avg_los
Elective,F,5816,4.9
Elective,M,5712,5.1
Elective,X,5576,5.0
Emergency,F,5670,4.9
Emergency,M,5654,5.0
Emergency,X,5760,5.0
Maternity,F,1950,4.7
Maternity,M,1940,5.0
Maternity,X,2018,4.8
Newborn,F,1295,4.9


In [0]:
# 7d. Aggregation on the ED presentations dataset
display(
    df_ed
    .filter(F.col("triage_category").isNotNull())
    .groupBy("triage_category")
    .agg(
        F.count("*").alias("presentations"),
        F.round(F.avg("wait_time_minutes"), 1).alias("avg_wait_mins"),
        F.round(F.avg("total_ed_time_minutes"), 1).alias("avg_total_ed_mins"),
        F.round(F.avg("met_4hour_target"), 2).alias("pct_met_4hr_target")
    )
    .orderBy("triage_category")
)

triage_category,presentations,avg_wait_mins,avg_total_ed_mins,pct_met_4hr_target
1.0,649,-1.5,107.3,0.93
2.0,3475,17.5,127.4,0.93
3.0,10319,58.8,167.7,0.85
4.0,13521,120.3,228.5,0.6
5.0,6313,242.4,351.6,0.24


---
## 8️⃣ Joins

Joins combine DataFrames just like SQL. The key difference:
in PySpark you specify the join condition as an **expression**, not just column names.

| Join Type | What It Returns |
|-----------|----------------|
| `inner`   | Only matching rows from both sides |
| `left`    | All rows from left + matches from right |
| `anti`    | Rows from left that have **no** match on right |

In [0]:
# 8a. INNER JOIN — Admissions with hospital details
df_adm_hospital = (
    df_admissions
    .join(df_hospitals, on="hospital_id", how="inner")
    .select(
        "admission_id",
        "hospital_name",
        "hospital_tier",
        "admission_type",
        "length_of_stay_days",
        "abf_price_weight"
    )
)
display(df_adm_hospital.limit(10))
print(f"Matched rows: {df_adm_hospital.count():,}")

admission_id,hospital_name,hospital_tier,admission_type,length_of_stay_days,abf_price_weight
ADM00000001,Westmead Hospital,Tertiary,Elective,5,1.5736
ADM00000002,Shoalhaven Hospital,District,Newborn,15,0.8488
ADM00000003,Blue Mountains Hospital,Major,Elective,16,0.7199
ADM00000004,Gosford Hospital,Major,Elective,5,0.7218
ADM00000005,Prince of Wales Hospital,Tertiary,Emergency,4,0.299
ADM00000006,Shellharbour Hospital,Major,Elective,1,1.246
ADM00000007,Prince of Wales Hospital,Tertiary,Emergency,1,2.7898
ADM00000008,Griffith Hospital,Community,Elective,2,0.3689
ADM00000009,Dubbo Hospital,Major,Emergency,1,0.4468
ADM00000010,Lismore Base Hospital,Major,Elective,4,0.8153


Matched rows: 51,460


In [0]:
# 8b. LEFT JOIN — All LHDs with their admission counts (even if zero)
df_lhd_admissions = (
    df_lhd
    .join(
        df_admissions.groupBy("lhd_code").agg(F.count("*").alias("total_admissions")),
        on="lhd_code",
        how="left"
    )
    .select("lhd_name", "region_type", "population_served", "total_admissions")
    .fillna({"total_admissions": 0})
    .orderBy("total_admissions", ascending=False)
)
display(df_lhd_admissions)

lhd_name,region_type,population_served,total_admissions
Western NSW,Remote,280000,5758
Southern NSW,Rural,210000,4365
Mid North Coast,Rural,220000,4325
Illawarra Shoalhaven,Regional,410000,4171
South Western Sydney,Metropolitan,1020000,3620
Sydney Metropolitan,Metropolitan,820000,3267
Northern NSW,Rural,310000,3254
Northern Sydney,Metropolitan,890000,3239
South Eastern Sydney,Metropolitan,930000,3213
Western Sydney,Metropolitan,960000,3185


In [0]:
# 8c. ANTI JOIN — Find admissions with invalid hospital IDs
#     (hospitals in admissions that DON'T exist in dim_hospitals)
df_invalid_hospitals = (
    df_admissions
    .join(df_hospitals, on="hospital_id", how="anti")
    .select("admission_id", "hospital_id")
    .distinct()
)
print(f"Admissions with invalid hospital_id: {df_invalid_hospitals.count():,}")
display(df_invalid_hospitals.limit(10))

Admissions with invalid hospital_id: 40


admission_id,hospital_id
ADM00006549,H9999
ADM00002178,H9999
ADM00015231,H9999
ADM00001283,H9999
ADM00018738,H9999
ADM00006261,H9999
ADM00000407,H9999
ADM00006822,H9999
ADM00006585,H9999
ADM00003967,H9999


In [0]:
# 8d. Multi-table join — Admissions + Hospitals + LHD
df_enriched = (
    df_admissions
    .join(df_hospitals, on="hospital_id", how="inner")
    .join(df_lhd, on="lhd_code", how="inner")
    .select(
        "admission_id",
        "lhd_name",
        "region_type",
        "hospital_name",
        "hospital_tier",
        "admission_type",
        "patient_age",
        "length_of_stay_days",
        "abf_price_weight"
    )
)
print(f"Enriched dataset: {df_enriched.count():,} rows x {len(df_enriched.columns)} columns")
display(df_enriched.limit(10))

Enriched dataset: 51,460 rows x 9 columns


admission_id,lhd_name,region_type,hospital_name,hospital_tier,admission_type,patient_age,length_of_stay_days,abf_price_weight
ADM00000001,Western Sydney,Metropolitan,Westmead Hospital,Tertiary,Elective,48.0,5,1.5736
ADM00000002,Illawarra Shoalhaven,Regional,Shoalhaven Hospital,District,Newborn,46.0,15,0.8488
ADM00000003,Nepean Blue Mountains,Regional,Blue Mountains Hospital,Major,Elective,68.0,16,0.7199
ADM00000004,Central Coast,Regional,Gosford Hospital,Major,Elective,41.0,5,0.7218
ADM00000005,South Eastern Sydney,Metropolitan,Prince of Wales Hospital,Tertiary,Emergency,9.0,4,0.299
ADM00000006,Illawarra Shoalhaven,Regional,Shellharbour Hospital,Major,Elective,29.0,1,1.246
ADM00000007,South Eastern Sydney,Metropolitan,Prince of Wales Hospital,Tertiary,Emergency,20.0,1,2.7898
ADM00000008,Murrumbidgee,Rural,Griffith Hospital,Community,Elective,53.0,2,0.3689
ADM00000009,Western NSW,Remote,Dubbo Hospital,Major,Emergency,54.0,1,0.4468
ADM00000010,Northern NSW,Rural,Lismore Base Hospital,Major,Elective,38.0,4,0.8153


---
## 9️⃣ Window Functions

Window functions calculate values **across a set of rows** related to the current row —
without collapsing them like `groupBy`. Think: "rank within each group" or "running total".

**Pattern:** `F.function().over(Window.partitionBy(...).orderBy(...))`

In [0]:
# 9a. RANK — Rank hospitals by admission count within each LHD
window_lhd = Window.partitionBy("lhd_code").orderBy(F.desc("total_admissions"))

df_hospital_ranked = (
    df_admissions
    .groupBy("hospital_id", "lhd_code")
    .agg(F.count("*").alias("total_admissions"))
    .join(df_hospitals.select("hospital_id", "hospital_name"), on="hospital_id")
    .join(df_lhd.select("lhd_code", "lhd_name"), on="lhd_code")
    .withColumn("rank_in_lhd", F.rank().over(window_lhd))
    .filter(F.col("rank_in_lhd") <= 3)  # Top 3 per LHD
    .orderBy("lhd_name", "rank_in_lhd")
    .select("lhd_name", "rank_in_lhd", "hospital_name", "total_admissions")
)

display(df_hospital_ranked)

lhd_name,rank_in_lhd,hospital_name,total_admissions
Central Coast,1,Wyong Hospital,1504
Central Coast,2,Gosford Hospital,1400
Far West,1,Broken Hill Hospital,1432
Hunter New England,1,John Hunter Hospital,1423
Hunter New England,2,Calvary Mater Newcastle,342
Hunter New England,3,Armidale Hospital,323
Illawarra Shoalhaven,1,Shoalhaven Hospital,1442
Illawarra Shoalhaven,2,Shellharbour Hospital,1368
Illawarra Shoalhaven,3,Wollongong Hospital,1357
Mid North Coast,1,Kempsey Hospital,1490


In [0]:
# 9b. ROW_NUMBER — Deduplicate: keep only one record per patient per hospital
window_dedup = (
    Window
    .partitionBy("patient_id", "hospital_id")
    .orderBy(F.desc("admission_date"))
)

df_latest_admission = (
    df_admissions
    .withColumn("rn", F.row_number().over(window_dedup))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
print(f"Before dedup: {df_admissions.count():,}")
print(f"After dedup:  {df_latest_admission.count():,}")

Before dedup: 51,500
After dedup:  49,967


In [0]:
# 9c. Running / Cumulative aggregation — average LOS by admission type over age
window_running = (
    Window
    .partitionBy("admission_type")
    .orderBy("patient_age")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_running_avg = (
    df_admissions
    .filter(F.col("patient_age").isNotNull() & (F.col("length_of_stay_days") > 0))
    .withColumn("running_avg_los", F.round(F.avg("length_of_stay_days").over(window_running), 2))
    .select("admission_type", "patient_age", "length_of_stay_days", "running_avg_los")
)

display(df_running_avg.filter(F.col("admission_type") == "Emergency").limit(20))

admission_type,patient_age,length_of_stay_days,running_avg_los
Emergency,0.0,14,14.0
Emergency,0.0,8,11.0
Emergency,0.0,5,9.0
Emergency,0.0,2,7.25
Emergency,0.0,5,6.8
Emergency,0.0,3,6.17
Emergency,0.0,3,5.71
Emergency,0.0,2,5.25
Emergency,0.0,6,5.33
Emergency,0.0,3,5.1


In [0]:
# 9d. LAG / LEAD — Compare with previous/next row
window_patient = Window.partitionBy("patient_id").orderBy("admission_date")

df_readmission = (
    df_admissions
    .withColumn("prev_separation", F.lag("separation_date").over(window_patient))
    .withColumn(
        "days_since_last_admission",
        F.datediff(
            F.expr('try_cast(admission_date as date)'),
            F.col("prev_separation")
        )
    )
    .filter(F.col("days_since_last_admission").isNotNull())
    .filter(F.col("days_since_last_admission").between(0, 30))
    .select("patient_id", "admission_date", "prev_separation", "days_since_last_admission", "hospital_id")
)

print(f"Potential readmissions (within 30 days): {df_readmission.count():,}")
display(df_readmission.limit(10))

Potential readmissions (within 30 days): 922


patient_id,admission_date,prev_separation,days_since_last_admission,hospital_id
P101647,2025-01-27,2025-01-27,0,H0028
P102265,2025-02-19,2025-02-19,0,H0018
P102561,2025-03-11,2025-02-12,27,H0036
P104508,2025-02-03,2025-02-03,0,H0037
P106718,2025-02-13,2025-01-15,29,H0030
P107095,2025-02-23,2025-02-23,0,H0014
P107157,2025-03-23,2025-02-24,27,H0032
P107215,2025-02-16,2025-01-23,24,H0044
P108787,2025-02-21,2025-01-31,21,H0001
P110281,2025-02-01,2025-02-01,0,H0035


---
## 🥈 Building the Silver Layer (PySpark)

Now we apply everything we've learned to **clean the Bronze data** and write
a production-quality Silver table into Unity Catalog.

In [0]:
# Silver: Clean hospital admissions
df_silver_admissions = (
    df_admissions

    # 1. Remove duplicates (suffixed with _DUP)
    .filter(~F.col("admission_id").like("%_DUP"))

    # 2. Remove invalid hospital references
    .filter(F.col("hospital_id") != "H9999")

    # 3. Drop rows with critical nulls
    .filter(
        F.col("drg_code").isNotNull() &
        F.col("patient_age").isNotNull() &
        F.col("patient_gender").isNotNull()
    )

    # 4. Fix negative length of stay
    .withColumn(
        "length_of_stay_days",
        F.when(F.col("length_of_stay_days") < 0, F.lit(0))
         .otherwise(F.col("length_of_stay_days"))
    )

    # 5. Add derived columns
    .withColumn("los_category",
        F.when(F.col("length_of_stay_days") <= 1, "Day Case")
         .when(F.col("length_of_stay_days") <= 3, "Short Stay")
         .when(F.col("length_of_stay_days") <= 7, "Medium Stay")
         .when(F.col("length_of_stay_days") <= 14, "Long Stay")
         .otherwise("Extended Stay")
    )

    # 6. Add audit column
    .withColumn("silver_loaded_at", F.current_timestamp())
)

print(f"Bronze rows:  {df_admissions.count():,}")
print(f"Silver rows:  {df_silver_admissions.count():,}")
print(f"Rows removed: {df_admissions.count() - df_silver_admissions.count():,}")

Bronze rows:  51,500
Silver rows:  42,859
Rows removed: 8,641


In [0]:
# Write to Silver layer in Unity Catalog
(
    df_silver_admissions
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("zoteo_health.silver.hospital_admissions_pyspark")
)

print("✅ Silver table written: zoteo_health.silver.hospital_admissions_pyspark")

✅ Silver table written: zoteo_health.silver.hospital_admissions_pyspark


In [0]:
# Similarly, clean ED presentations
df_silver_ed = (
    df_ed
    .filter(~F.col("ed_presentation_id").like("%_DUP"))
    .filter(F.col("triage_category").isNotNull())
    .filter(F.col("wait_time_minutes") >= 0)
    .filter(F.col("total_ed_time_minutes") > 0)
    .withColumn(
        "triage_description",
        F.when(F.col("triage_category") == 1, "Resuscitation")
         .when(F.col("triage_category") == 2, "Emergency")
         .when(F.col("triage_category") == 3, "Urgent")
         .when(F.col("triage_category") == 4, "Semi-urgent")
         .when(F.col("triage_category") == 5, "Non-urgent")
         .otherwise("Unknown")
    )
    .withColumn("silver_loaded_at", F.current_timestamp())
)

(
    df_silver_ed
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("zoteo_health.silver.ed_presentations_pyspark")
)

print(f"✅ Silver ED table written: {df_silver_ed.count():,} rows")

✅ Silver ED table written: 33,541 rows


---
## 🥇 Building the Gold Layer — Analytics-Ready Outputs

The Gold layer is where all our PySpark skills come together.
We create **business-ready** aggregated datasets that analysts and dashboards consume.

### Gold Table 1: Hospital Performance Scorecard

In [0]:
# Read from Silver
df_silver = spark.table("zoteo_health.silver.hospital_admissions_pyspark")

# Build the scorecard
df_gold_hospital = (
    df_silver
    .join(df_hospitals, on="hospital_id", how="inner")
    .join(df_lhd, on="lhd_code", how="inner")
    .groupBy(
        "lhd_name", "region_type", "hospital_name", "hospital_tier", "total_beds"
    )
    .agg(
        F.count("*").alias("total_admissions"),
        F.countDistinct("patient_id").alias("unique_patients"),
        F.round(F.avg("length_of_stay_days"), 1).alias("avg_los_days"),
        F.round(F.sum("abf_price_weight"), 2).alias("total_abf_weight"),
        F.round(F.avg("abf_price_weight"), 3).alias("avg_abf_weight"),

        # Admission type breakdown
        F.count(F.when(F.col("admission_type") == "Emergency", True)).alias("emergency_count"),
        F.count(F.when(F.col("admission_type") == "Elective", True)).alias("elective_count"),

        # Age demographics
        F.round(F.avg("patient_age"), 1).alias("avg_patient_age"),
        F.count(F.when(F.col("patient_age") >= 65, True)).alias("patients_65_plus"),

        # LOS distribution
        F.count(F.when(F.col("los_category") == "Day Case", True)).alias("day_cases"),
        F.count(F.when(F.col("los_category") == "Extended Stay", True)).alias("extended_stays"),
    )
    .withColumn("emergency_pct", F.round(F.col("emergency_count") / F.col("total_admissions") * 100, 1))
    .withColumn("bed_utilisation_proxy", F.round(F.col("total_admissions") / F.col("total_beds"), 1))
    .orderBy("total_admissions", ascending=False)
)

# Write to Gold
(
    df_gold_hospital
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("zoteo_health.gold.hospital_performance_scorecard")
)

print("✅ Gold table written: zoteo_health.gold.hospital_performance_scorecard")
display(df_gold_hospital.limit(15))

✅ Gold table written: zoteo_health.gold.hospital_performance_scorecard


lhd_name,region_type,hospital_name,hospital_tier,total_beds,total_admissions,unique_patients,avg_los_days,total_abf_weight,avg_abf_weight,emergency_count,elective_count,avg_patient_age,patients_65_plus,day_cases,extended_stays,emergency_pct,bed_utilisation_proxy
Western Sydney,Metropolitan,Auburn Hospital,District,166,1238,1235,4.8,1506.09,1.217,419,426,51.6,348,357,75,33.8,7.5
Southern NSW,Rural,Batemans Bay Hospital,District,104,1238,1236,5.1,1625.22,1.313,442,431,52.2,353,377,93,35.7,11.9
Northern Sydney,Metropolitan,Manly Hospital,District,225,1238,1238,5.3,1607.33,1.298,437,424,50.6,336,369,101,35.3,5.5
Northern NSW,Rural,Lismore Base Hospital,Major,492,1238,1237,5.1,1614.5,1.304,413,470,51.8,348,359,89,33.4,2.5
Western NSW,Remote,Bathurst Hospital,Major,250,1237,1237,5.0,1598.77,1.292,435,435,50.8,335,376,88,35.2,4.9
Southern NSW,Rural,Moruya Hospital,District,93,1231,1231,4.6,1563.57,1.27,433,424,51.9,366,401,71,35.2,13.2
Central Coast,Regional,Wyong Hospital,District,172,1229,1229,4.8,1470.65,1.197,421,428,51.1,319,395,70,34.3,7.1
Mid North Coast,Rural,Kempsey Hospital,Major,268,1226,1226,4.8,1597.71,1.303,452,425,52.0,354,377,72,36.9,4.6
Sydney Metropolitan,Metropolitan,Royal Prince Albert Hospital,Tertiary,557,1218,1217,5.0,1562.49,1.283,434,434,51.0,317,387,85,35.6,2.2
Far West,Remote,Broken Hill Hospital,Major,305,1210,1210,4.9,1553.89,1.284,411,413,51.8,347,383,66,34.0,4.0


### Gold Table 2: LHD Regional Comparison

In [0]:
df_gold_lhd = (
    df_silver
    .join(df_lhd, on="lhd_code", how="inner")
    .groupBy("lhd_code", "lhd_name", "region_type", "population_served", "annual_budget_aud_millions")
    .agg(
        F.count("*").alias("total_admissions"),
        F.countDistinct("patient_id").alias("unique_patients"),
        F.round(F.avg("length_of_stay_days"), 1).alias("avg_los"),
        F.round(F.sum("abf_price_weight"), 2).alias("total_abf_weight"),
        F.count(F.when(F.col("admission_type") == "Emergency", True)).alias("emergency_admissions"),
    )
    .withColumn("admissions_per_1000_pop",
        F.round(F.col("total_admissions") / F.col("population_served") * 1000, 1))
    .withColumn("cost_per_admission_proxy",
        F.when(F.col("total_admissions") == 0, None)
         .otherwise(F.round(F.col("annual_budget_aud_millions").cast("double") * 1000000 / F.col("total_admissions"), 2)))
    .orderBy("total_admissions", ascending=False)
)

(
    df_gold_lhd
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("zoteo_health.gold.lhd_regional_comparison")
)

print("✅ Gold table written: zoteo_health.gold.lhd_regional_comparison")
display(df_gold_lhd)

✅ Gold table written: zoteo_health.gold.lhd_regional_comparison


lhd_code,lhd_name,region_type,population_served,annual_budget_aud_millions,total_admissions,unique_patients,avg_los,total_abf_weight,emergency_admissions,admissions_per_1000_pop,cost_per_admission_proxy
LHD014,Western NSW,Remote,280000,560,4754,4739,4.9,6134.27,1648,17.0,117795.54
LHD012,Southern NSW,Rural,210000,420,3648,3639,4.9,4664.6,1307,17.4,115131.58
LHD011,Mid North Coast,Rural,220000,440,3554,3548,4.8,4613.86,1261,16.2,123804.16
LHD008,Illawarra Shoalhaven,Regional,410000,820,3533,3526,4.9,4528.47,1247,8.6,232097.37
LHD003,South Western Sydney,Metropolitan,1020000,2250,2985,2980,5.1,3853.3,1095,2.9,753768.84
LHD001,Sydney Metropolitan,Metropolitan,820000,1850,2730,2727,5.2,3516.72,960,3.3,677655.68
LHD010,Northern NSW,Rural,310000,610,2726,2717,5.1,3542.18,959,8.8,223771.09
LHD005,Northern Sydney,Metropolitan,890000,1950,2691,2686,5.0,3494.68,943,3.0,724637.68
LHD013,Murrumbidgee,Rural,240000,480,2678,2674,4.8,3549.63,917,11.2,179238.24
LHD002,Western Sydney,Metropolitan,960000,2100,2667,2663,4.8,3378.02,931,2.8,787401.57


### Gold Table 3: ED Performance by Triage Category

In [0]:
df_silver_ed = spark.table("zoteo_health.silver.ed_presentations_pyspark")

df_gold_ed = (
    df_silver_ed
    .join(df_hospitals.select("hospital_id", "hospital_name", "hospital_tier"), on="hospital_id")
    .join(df_lhd.select("lhd_code", "lhd_name", "region_type"), on="lhd_code")
    .groupBy("lhd_name", "region_type", "triage_category", "triage_description")
    .agg(
        F.count("*").alias("total_presentations"),
        F.round(F.avg("wait_time_minutes"), 1).alias("avg_wait_mins"),
        F.round(F.avg("total_ed_time_minutes"), 1).alias("avg_ed_time_mins"),
        F.round(F.avg("met_4hour_target") * 100, 1).alias("pct_met_4hr_target"),

        # Disposition breakdown
        F.count(F.when(F.col("ed_disposition") == "Admitted", True)).alias("admitted_count"),
        F.count(F.when(F.col("ed_disposition") == "Discharged", True)).alias("discharged_count"),

        # Arrival mode
        F.count(F.when(F.col("arrival_mode") == "Ambulance", True)).alias("ambulance_arrivals"),
    )
    .withColumn("admission_rate_pct",
        F.round(F.col("admitted_count") / F.col("total_presentations") * 100, 1))
    .orderBy("lhd_name", "triage_category")
)

(
    df_gold_ed
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("zoteo_health.gold.ed_triage_performance")
)

print("✅ Gold table written: zoteo_health.gold.ed_triage_performance")
display(df_gold_ed.limit(20))

✅ Gold table written: zoteo_health.gold.ed_triage_performance


lhd_name,region_type,triage_category,triage_description,total_presentations,avg_wait_mins,avg_ed_time_mins,pct_met_4hr_target,admitted_count,discharged_count,ambulance_arrivals,admission_rate_pct
Central Coast,Regional,1.0,Resuscitation,27,0.0,97.0,100.0,7,9,6,25.9
Central Coast,Regional,2.0,Emergency,185,19.8,122.5,93.0,61,82,30,33.0
Central Coast,Regional,3.0,Urgent,457,59.9,161.2,88.6,139,202,85,30.4
Central Coast,Regional,4.0,Semi-urgent,648,120.3,227.0,59.3,216,272,125,33.3
Central Coast,Regional,5.0,Non-urgent,293,253.0,364.3,20.1,82,130,53,28.0
Far West,Remote,1.0,Resuscitation,14,0.0,85.8,100.0,1,6,4,7.1
Far West,Remote,2.0,Emergency,68,24.6,148.0,88.2,22,24,17,32.4
Far West,Remote,3.0,Urgent,254,75.3,184.9,80.3,76,114,49,29.9
Far West,Remote,4.0,Semi-urgent,318,146.2,250.1,48.7,90,146,66,28.3
Far West,Remote,5.0,Non-urgent,140,276.4,386.0,17.9,35,70,28,25.0


### Gold View 4: Funding vs Activity Analysis

This one we create as a **VIEW** — no data duplication, always reads the latest Silver + Bronze data.

In [0]:
df_gold_funding = (
    df_funding
    .join(df_lhd.select("lhd_code", "lhd_name", "region_type"), on="lhd_code")
    .join(
        df_silver
        .groupBy("lhd_code")
        .agg(
            F.count("*").alias("total_silver_admissions"),
            F.round(F.sum("abf_price_weight"), 2).alias("total_silver_abf_weight")
        ),
        on="lhd_code",
        how="left"
    )
    .withColumn("nwau_variance_pct",
        F.round((F.col("actual_activity_nwau") - F.col("benchmark_nwau")) / F.col("benchmark_nwau") * 100, 1))
    .withColumn("is_over_performing",
        F.when(F.col("actual_activity_nwau") > F.col("benchmark_nwau"), "Yes").otherwise("No"))
    .select(
        "lhd_name", "region_type", "financial_year", "quarter",
        "allocated_budget_aud_millions", "actual_activity_nwau", "benchmark_nwau",
        "nwau_variance_pct", "is_over_performing", "funding_adjustment_aud_millions",
        "reporting_compliance_pct", "total_silver_admissions", "total_silver_abf_weight"
    )
)

# Write the analysis as a permanent table
(
    df_gold_funding
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("zoteo_health.gold.funding_analysis")
)

# Create the Gold view from the permanent table
spark.sql("""
    CREATE OR REPLACE VIEW zoteo_health.gold.vw_funding_vs_activity AS SELECT * FROM zoteo_health.gold.funding_analysis
""")

print("✅ Gold view created: zoteo_health.gold.vw_funding_vs_activity")
display(df_gold_funding.orderBy("lhd_name", "quarter"))

✅ Gold view created: zoteo_health.gold.vw_funding_vs_activity


lhd_name,region_type,financial_year,quarter,allocated_budget_aud_millions,actual_activity_nwau,benchmark_nwau,nwau_variance_pct,is_over_performing,funding_adjustment_aud_millions,reporting_compliance_pct,total_silver_admissions,total_silver_abf_weight
Central Coast,Regional,FY2024-25,Q1,166.13,85163,85000,0.2,Yes,-3.09,91.0,2383,2928.16
Central Coast,Regional,FY2024-25,Q2,161.97,70812,85000,-16.7,No,-6.42,97.3,2383,2928.16
Central Coast,Regional,FY2024-25,Q3,162.43,89205,85000,4.9,Yes,-6.05,98.7,2383,2928.16
Central Coast,Regional,FY2024-25,Q4,189.4,86073,85000,1.3,Yes,15.52,99.1,2383,2928.16
Far West,Remote,FY2024-25,Q1,24.49,12972,11875,9.2,Yes,0.59,100.0,1210,1553.89
Far West,Remote,FY2024-25,Q2,22.77,12572,11875,5.9,Yes,-0.79,89.9,1210,1553.89
Far West,Remote,FY2024-25,Q3,23.38,11506,11875,-3.1,No,-0.29,95.3,1210,1553.89
Far West,Remote,FY2024-25,Q4,24.05,12847,11875,8.2,Yes,0.24,90.3,1210,1553.89
Hunter New England,Regional,FY2024-25,Q1,463.36,219292,237500,-7.7,No,-9.31,88.4,2004,2516.54
Hunter New England,Regional,FY2024-25,Q2,493.0,250260,237500,5.4,Yes,14.4,89.3,2004,2516.54


---
## 1️⃣1️⃣ Bonus: Convert to Pandas for Visualisation

PySpark DataFrames are distributed — they live across a cluster.
To use matplotlib, seaborn, or plotly, convert to Pandas first.

⚠️ **Only do this on aggregated/small datasets** — never `.toPandas()` on millions of rows.

In [0]:
import pandas as pd

# Convert Gold scorecard to Pandas
pdf_hospital = df_gold_hospital.toPandas()

print(f"Pandas DataFrame: {pdf_hospital.shape[0]} rows x {pdf_hospital.shape[1]} columns")
print(f"Memory usage: {pdf_hospital.memory_usage(deep=True).sum() / 1024:.1f} KB")
display(pdf_hospital.head())

Pandas DataFrame: 45 rows x 18 columns
Memory usage: 17.1 KB


lhd_name,region_type,hospital_name,hospital_tier,total_beds,total_admissions,unique_patients,avg_los_days,total_abf_weight,avg_abf_weight,emergency_count,elective_count,avg_patient_age,patients_65_plus,day_cases,extended_stays,emergency_pct,bed_utilisation_proxy
Southern NSW,Rural,Batemans Bay Hospital,District,104,1238,1236,5.1,1625.22,1.313,442,431,52.2,353,377,93,35.7,11.9
Northern NSW,Rural,Lismore Base Hospital,Major,492,1238,1237,5.1,1614.5,1.304,413,470,51.8,348,359,89,33.4,2.5
Western Sydney,Metropolitan,Auburn Hospital,District,166,1238,1235,4.8,1506.09,1.217,419,426,51.6,348,357,75,33.8,7.5
Northern Sydney,Metropolitan,Manly Hospital,District,225,1238,1238,5.3,1607.33,1.298,437,424,50.6,336,369,101,35.3,5.5
Western NSW,Remote,Bathurst Hospital,Major,250,1237,1237,5.0,1598.77,1.292,435,435,50.8,335,376,88,35.2,4.9


In [0]:
# You can also go from Pandas back to PySpark
df_back_to_spark = spark.createDataFrame(pdf_hospital)
print(f"Back to PySpark: {df_back_to_spark.count()} rows")

---
## ✅ Final Check — What We Built

Let's verify everything in Unity Catalog:

In [0]:
# List all objects we created
print("=" * 60)
print("SILVER LAYER — Cleaned Tables")
print("=" * 60)
for table in spark.catalog.listTables("silver"):
    print(f"  📋 {table.name:40s} ({table.tableType})")

print()
print("=" * 60)
print("GOLD LAYER — Analytics-Ready Outputs")
print("=" * 60)
for table in spark.catalog.listTables("gold"):
    print(f"  {'📊' if table.tableType == 'VIEW' else '📋'} {table.name:40s} ({table.tableType})")

SILVER LAYER — Cleaned Tables
  📋 ed_presentations_pyspark                 (MANAGED)
  📋 hospital_admissions                      (MANAGED)
  📋 hospital_admissions_pyspark              (MANAGED)
  📋 tmp_funding_analysis                     (TEMPORARY)

GOLD LAYER — Analytics-Ready Outputs
  📋 ed_triage_performance                    (MANAGED)
  📋 funding_analysis                         (MANAGED)
  📋 hospital_performance_scorecard           (MANAGED)
  📋 lhd_regional_comparison                  (MANAGED)
  📊 vw_admissions_by_hospital                (VIEW)
  📊 vw_funding_vs_activity                   (VIEW)
  📊 vw_lhd_performance                       (VIEW)
  📋 tmp_funding_analysis                     (TEMPORARY)


---
## 📚 PySpark Cheat Sheet — Quick Reference

| Operation | PySpark | SQL Equivalent |
|-----------|---------|----------------|
| Read table | `spark.table("catalog.schema.table")` | `SELECT * FROM table` |
| Select columns | `df.select("col1", "col2")` | `SELECT col1, col2` |
| Filter rows | `df.filter(F.col("age") > 18)` | `WHERE age > 18` |
| Add column | `df.withColumn("new", F.lit("x"))` | `SELECT *, 'x' AS new` |
| CASE WHEN | `F.when(cond, val).otherwise(val)` | `CASE WHEN ... THEN ... END` |
| Group & aggregate | `df.groupBy("col").agg(F.count("*"))` | `GROUP BY col` |
| Join | `df1.join(df2, on="key", how="inner")` | `JOIN df2 ON key` |
| Window function | `F.rank().over(Window.partitionBy(...))` | `RANK() OVER (PARTITION BY ...)` |
| Remove nulls | `df.dropna(subset=["col"])` | `WHERE col IS NOT NULL` |
| Fill nulls | `df.fillna({"col": "default"})` | `COALESCE(col, 'default')` |
| Write table | `df.write.saveAsTable("catalog.schema.t")` | `CREATE TABLE AS SELECT` |
| To Pandas | `df.toPandas()` | *(N/A)* |